#  Phase 1: Data Overview - Electronics Sales Analysis

## 🧩 Tasks Checklist:
- [ ] **Step 1**: Load the data - Ensure it reads cleanly without encoding issues
- [ ] **Step 2**: Inspect shape - Know how many rows and columns 
- [ ] **Step 3**: Check column names & data types - Identify categorical vs numerical features
- [ ] **Step 4**: Preview first rows - Verify content and structure
- [ ] **Step 5**: Check for missing/null values - Detect incomplete records early
- [ ] **Step 6**: Check for duplicates - Remove repeated rows if any
- [ ] **Step 7**: Validate date parsing - Ensure "Purchase Date" is recognized as datetime

---
**Dataset**: Electronic Sales (Sep 2023 - Sep 2024)  
**Expected Records**: ~20,000 transactions

In [3]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set display options for better output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print(" Libraries imported successfully!")
print(f"📅 Analysis started on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

 Libraries imported successfully!
📅 Analysis started on: 2025-10-13 20:52:30


In [4]:
# Setup figure saving configuration
import os
from pathlib import Path

# Create figure directories if they don't exist
FIGURE_DIR = Path("../../outputs/figures")
FIGURE_SUBDIRS = {
    'exploratory': FIGURE_DIR / 'exploratory',
    'correlations': FIGURE_DIR / 'correlations', 
    'distributions': FIGURE_DIR / 'distributions',
    'model_performance': FIGURE_DIR / 'model_performance',
    'business_insights': FIGURE_DIR / 'business_insights'
}

# Create all directories
for subdir_name, subdir_path in FIGURE_SUBDIRS.items():
    subdir_path.mkdir(parents=True, exist_ok=True)
    print(f"📁 Created: {subdir_path}")

# Configure matplotlib for high-quality figure saving
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.format'] = 'png'
plt.rcParams['savefig.bbox'] = 'tight'

# Helper function to save figures
def save_figure(fig, filename, category='exploratory', formats=['png', 'pdf']):
    """Save figure to appropriate directory with multiple formats"""
    save_dir = FIGURE_SUBDIRS[category]
    
    for fmt in formats:
        filepath = save_dir / f"{filename}.{fmt}"
        fig.savefig(filepath, format=fmt, dpi=300, bbox_inches='tight')
        print(f"💾 Saved: {filepath}")

print("\n Figure saving configuration complete!")
print(f" Main figures directory: {FIGURE_DIR.absolute()}")
print(f"📈 Available categories: {list(FIGURE_SUBDIRS.keys())}")

📁 Created: ..\..\outputs\figures\exploratory
📁 Created: ..\..\outputs\figures\correlations
📁 Created: ..\..\outputs\figures\distributions
📁 Created: ..\..\outputs\figures\model_performance
📁 Created: ..\..\outputs\figures\business_insights

 Figure saving configuration complete!
 Main figures directory: c:\Users\hatim\OneDrive\سطح المكتب\iau\25.26\Data_Mining\Project\DataMining\notebooks\expalortation\..\..\outputs\figures
📈 Available categories: ['exploratory', 'correlations', 'distributions', 'model_performance', 'business_insights']


## 📁 Step 1: Load the Data
**Goal**: Ensure the CSV file reads cleanly without encoding issues

In [5]:
# Step 1: Load the data with error handling
try:
    # Define file path (going up two levels from notebooks/expalortation/)
    data_path = "../../data/raw/Electronic_sales_Sep2023-Sep2024.csv"
    
    # Load the dataset
    df = pd.read_csv(data_path)
    
    print(" Step 1 COMPLETED: Data loaded successfully!")
    print(f"📄 File path: {data_path}")
    print(f" Initial data loaded: {len(df)} records")
    
except FileNotFoundError:
    print("❌ Error: CSV file not found. Check the file path.")
except UnicodeDecodeError:
    print("⚠️  Encoding issue detected. Trying with different encoding...")
    try:
        df = pd.read_csv(data_path, encoding='latin-1')
        print(" Data loaded with latin-1 encoding")
    except:
        print("❌ Failed to load with alternative encoding")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

 Step 1 COMPLETED: Data loaded successfully!
📄 File path: ../../data/raw/Electronic_sales_Sep2023-Sep2024.csv
 Initial data loaded: 20000 records


## 📐 Step 2: Inspect Shape
**Goal**: Know how many rows and columns we have

In [6]:
# Step 2: Inspect dataset shape
print(" Step 2 COMPLETED: Dataset Shape Analysis")
print("=" * 50)
print(f" Dataset Shape: {df.shape}")
print(f"📈 Total Records (Rows): {df.shape[0]:,}")
print(f"📋 Total Features (Columns): {df.shape[1]}")
print(f"💾 Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Quick size validation
expected_records = 20000
actual_records = df.shape[0]
if actual_records == expected_records:
    print(f" Record count matches expectation: {expected_records:,}")
else:
    print(f"⚠️  Record count differs from expected:")
    print(f"   Expected: {expected_records:,}")
    print(f"   Actual: {actual_records:,}")
    print(f"   Difference: {abs(actual_records - expected_records):,}")

 Step 2 COMPLETED: Dataset Shape Analysis
 Dataset Shape: (20000, 16)
📈 Total Records (Rows): 20,000
📋 Total Features (Columns): 16
💾 Memory Usage: 10.90 MB
 Record count matches expectation: 20,000


## 🏷️ Step 3: Column Names & Data Types
**Goal**: Identify categorical vs numerical features

In [8]:
# Step 3: Analyze column names and data types
print(" Step 3 COMPLETED: Column Names & Data Types Analysis")
print("=" * 60)

# Display basic info
print("📋 DATASET INFO:")
df.info(memory_usage='deep')

print("\n" + "=" * 60)
print("🏷️  COLUMN ANALYSIS:")

# Categorize columns by data type
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
datetime_cols = df.select_dtypes(include=['datetime']).columns.tolist()

print(f"\n NUMERIC COLUMNS ({len(numeric_cols)}):")
for i, col in enumerate(numeric_cols, 1):
    print(f"   {i}. {col} ({df[col].dtype})")

print(f"\n🏷️  CATEGORICAL/TEXT COLUMNS ({len(categorical_cols)}):")
for i, col in enumerate(categorical_cols, 1):
    unique_count = df[col].nunique()
    print(f"   {i}. {col} ({df[col].dtype}) - {unique_count} unique values")

if datetime_cols:
    print(f"\n📅 DATETIME COLUMNS ({len(datetime_cols)}):")
    for i, col in enumerate(datetime_cols, 1):
        print(f"   {i}. {col} ({df[col].dtype})")
else:
    print(f"\n📅 DATETIME COLUMNS (0): No datetime columns detected yet")

print("\n" + "=" * 80)
print(" COMPREHENSIVE DATA TYPE CLASSIFICATION")
print("=" * 80)

# Simplified and fast classification function
def classify_data_type_fast(column_name, series):
    """Fast data type classification"""
    
    dtype = str(series.dtype)
    unique_count = series.nunique()
    null_count = series.isnull().sum()
    lower_name = column_name.lower()
    
    # Get first 3 sample values quickly
    sample_values = series.dropna().iloc[:3].tolist() if len(series) > 0 else []
    
    # Fast classification logic
    if series.dtype == 'bool':
        main_category, sub_category, measurement_scale = "Boolean", "Boolean", "Nominal Scale"
    elif unique_count == 2:
        main_category, sub_category, measurement_scale = "Binary", "Binary", "Nominal Scale"
    elif 'datetime' in dtype:
        main_category, sub_category, measurement_scale = "Date/Time", "DateTime", "Interval Scale"
    elif series.dtype in ['object']:
        if 'date' in lower_name and len(sample_values) > 0 and '-' in str(sample_values[0]):
            main_category, sub_category, measurement_scale = "Date/Time", "DateTime", "Interval Scale"
        elif any(x in lower_name for x in ['id', 'sku']):
            main_category, sub_category, measurement_scale = "Textual/String", "Identifier", "Nominal Scale"
        else:
            main_category, sub_category, measurement_scale = "Categorical", "Nominal", "Nominal Scale"
    elif series.dtype in ['int64', 'float64']:
        if 'id' in lower_name:
            main_category, sub_category, measurement_scale = "Textual/String", "Identifier", "Nominal Scale"
        elif 'rating' in lower_name or (unique_count <= 10 and series.min() >= 1 and series.max() <= 10 and 'quantity' not in lower_name):
            main_category, sub_category, measurement_scale = "Categorical", "Ordinal", "Ordinal Scale"
        elif any(x in lower_name for x in ['price', 'cost', 'amount', 'quantity', 'total', 'age']):
            main_category, sub_category, measurement_scale = "Numerical", "Ratio", "Ratio Scale"
        else:
            main_category, sub_category, measurement_scale = "Numerical", "Ratio", "Ratio Scale"
    else:
        main_category, sub_category, measurement_scale = "Unknown", "Unknown", "Unknown Scale"
    
    return {
        'Column': column_name,
        'Pandas_Type': dtype,
        'Unique_Values': unique_count,
        'Null_Count': null_count,
        'Null_%': round((null_count / len(series)) * 100, 1),
        'Main_Category': main_category,
        'Sub_Category': sub_category,
        'Measurement_Scale': measurement_scale,
        'Sample_Values': str(sample_values)[:30] + "..." if len(str(sample_values)) > 30 else str(sample_values)
    }

# Create analysis table quickly
print("Creating analysis table...")
analysis_results = []
for col in df.columns:
    result = classify_data_type_fast(col, df[col])
    analysis_results.append(result)

analysis_df = pd.DataFrame(analysis_results)

print("📋 COMPREHENSIVE DATA TYPE ANALYSIS TABLE:")
print("-" * 90)

# Display the table
display(analysis_df)

print("\n🔍 KEY CLASSIFICATIONS:")
print("-" * 50)
print(" Customer ID: Textual/String (Identifier)")
print(" Rating: Categorical Ordinal (1-5 ranking)")
print(" Quantity: Numerical Ratio (count data - quantitative)")
print(" Purchase Date: Date/Time (temporal data)")
print(" SKU: Textual/String (product identifier)")

print("\n🎯 CLASSIFICATION SUMMARY:")
print("-" * 50)
main_category_counts = analysis_df['Main_Category'].value_counts()
for category, count in main_category_counts.items():
    percentage = (count / len(analysis_df)) * 100
    print(f"   • {category}: {count} columns ({percentage:.1f}%)")

print("\n🎚️ MEASUREMENT SCALE DISTRIBUTION:")
print("-" * 50)
scale_counts = analysis_df['Measurement_Scale'].value_counts()
for scale, count in scale_counts.items():
    percentage = (count / len(analysis_df)) * 100
    print(f"   • {scale}: {count} columns ({percentage:.1f}%)")

print("\n📚 DATA TYPE HIERARCHY REFERENCE:")
print("-" * 50)
print(" Categorical → 🏷️ Nominal (no order) | 📈 Ordinal (with order)")
print("🔢 Numerical → 📏 Interval (no true zero) | 📐 Ratio (meaningful zero)")
print("⚡ Binary → Exactly 2 values | 📝 Textual/String → Text/IDs")
print("📅 Date/Time → Temporal data |  Boolean → True/False")

 Step 3 COMPLETED: Column Names & Data Types Analysis
📋 DATASET INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Customer ID        20000 non-null  int64  
 1   Age                20000 non-null  int64  
 2   Gender             19999 non-null  object 
 3   Loyalty Member     20000 non-null  object 
 4   Product Type       20000 non-null  object 
 5   SKU                20000 non-null  object 
 6   Rating             20000 non-null  int64  
 7   Order Status       20000 non-null  object 
 8   Payment Method     20000 non-null  object 
 9   Total Price        20000 non-null  float64
 10  Unit Price         20000 non-null  float64
 11  Quantity           20000 non-null  int64  
 12  Purchase Date      20000 non-null  object 
 13  Shipping Type      20000 non-null  object 
 14  Add-ons Purchased  15132 non-null  object 
 15  

,Column,Pandas_Type,Unique_Values,Null_Count,Null_%,Main_Category,Sub_Category,Measurement_Scale,Sample_Values
0,Customer ID,int64,12136,0,0.0,Textual/String,Identifier,Nominal Scale,"[1000, 1000, 1002]"
1,Age,int64,63,0,0.0,Numerical,Ratio,Ratio Scale,"[53, 53, 41]"
2,Gender,object,2,1,0.0,Binary,Binary,Nominal Scale,"['Male', 'Male', 'Male']"
3,Loyalty Member,object,2,0,0.0,Binary,Binary,Nominal Scale,"['No', 'No', 'No']"
4,Product Type,object,5,0,0.0,Categorical,Nominal,Nominal Scale,"['Smartphone', 'Tablet', 'Lapt..."
5,SKU,object,10,0,0.0,Textual/String,Identifier,Nominal Scale,"['SKU1004', 'SKU1002', 'SKU100..."
6,Rating,int64,5,0,0.0,Categorical,Ordinal,Ordinal Scale,"[2, 3, 3]"
7,Order Status,object,2,0,0.0,Binary,Binary,Nominal Scale,"['Cancelled', 'Completed', 'Co..."
8,Payment Method,object,6,0,0.0,Categorical,Nominal,Nominal Scale,"['Credit Card', 'Paypal', 'Cre..."
9,Total Price,float64,104,0,0.0,Numerical,Ratio,Ratio Scale,"[5538.33, 741.09, 1855.84]"



🔍 KEY CLASSIFICATIONS:
--------------------------------------------------
 Customer ID: Textual/String (Identifier)
 Rating: Categorical Ordinal (1-5 ranking)
 Quantity: Numerical Ratio (count data - quantitative)
 Purchase Date: Date/Time (temporal data)
 SKU: Textual/String (product identifier)

🎯 CLASSIFICATION SUMMARY:
--------------------------------------------------
   • Numerical: 5 columns (31.2%)
   • Categorical: 5 columns (31.2%)
   • Binary: 3 columns (18.8%)
   • Textual/String: 2 columns (12.5%)
   • Date/Time: 1 columns (6.2%)

🎚️ MEASUREMENT SCALE DISTRIBUTION:
--------------------------------------------------
   • Nominal Scale: 9 columns (56.2%)
   • Ratio Scale: 5 columns (31.2%)
   • Ordinal Scale: 1 columns (6.2%)
   • Interval Scale: 1 columns (6.2%)

📚 DATA TYPE HIERARCHY REFERENCE:
--------------------------------------------------
 Categorical → 🏷️ Nominal (no order) | 📈 Ordinal (with order)
🔢 Numerical → 📏 Interval (no true zero) | 📐 Ratio (meaningful zero)

## 👀 Step 4: Preview First Rows
**Goal**: Verify content and structure

In [8]:
# Step 4: Preview first rows
print(" Step 4 COMPLETED: Data Preview")
print("=" * 60)

print("👀 FIRST 5 ROWS:")
display(df.head())

print("\n🔚 LAST 3 ROWS:")
display(df.tail(3))

print("\n RANDOM SAMPLE (3 rows):")
display(df.sample(3, random_state=42))

print(f"\n📋 COLUMN NAMES ({len(df.columns)} total):")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2d}. {col}")

# Check for any obvious data quality issues in preview
print(f"\n🔍 QUICK DATA QUALITY CHECK:")
print(f"   • All columns present: {'' if len(df.columns) == 16 else '❌'}")
print(f"   • No completely empty columns: {'' if df.isnull().all().sum() == 0 else '❌'}")
print(f"   • Customer IDs look numeric: {'' if df['Customer ID'].dtype in ['int64', 'float64'] else '❌'}")

 Step 4 COMPLETED: Data Preview
👀 FIRST 5 ROWS:


,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
0,1000,53,Male,No,Smartphone,SKU1004,2,Cancelled,Credit Card,5538.33,791.19,7,2024-03-20,Standard,"Accessory,A...",40.21
1,1000,53,Male,No,Tablet,SKU1002,3,Completed,Paypal,741.09,247.03,3,2024-04-20,Overnight,Impulse Item,26.09
2,1002,41,Male,No,Laptop,SKU1005,3,Completed,Credit Card,1855.84,463.96,4,2023-10-17,Express,NaN,0.00
3,1002,41,Male,Yes,Smartphone,SKU1004,2,Completed,Cash,3164.76,791.19,4,2024-08-09,Overnight,Impulse Ite...,60.16
4,1003,75,Male,Yes,Smartphone,SKU1001,5,Completed,Cash,41.50,20.75,2,2024-05-21,Express,Accessory,35.56



🔚 LAST 3 ROWS:


,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
19997,19996,27,Female,No,Headphones,HDP456,4,Completed,Bank Transfer,1805.90,361.18,5,2024-08-26,Standard,Impulse Ite...,198.98
19998,19997,27,Male,No,Headphones,HDP456,1,Cancelled,Bank Transfer,2528.26,361.18,7,2024-01-06,Expedited,Extended Wa...,101.34
19999,19998,27,NaN,Yes,Laptop,LTP123,4,Completed,Bank Transfer,674.32,674.32,1,2024-01-29,Expedited,NaN,0.00



 RANDOM SAMPLE (3 rows):


,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
10650,11526,63,Female,No,Laptop,LTP123,4,Cancelled,PayPal,5394.56,674.32,8,2024-05-16,Expedited,NaN,0.00
2041,2805,35,Male,Yes,Tablet,SKU1002,3,Cancelled,Debit Card,1976.24,247.03,8,2024-04-25,Standard,NaN,0.00
8668,8801,49,Female,No,Smartwatch,SKU1003,3,Completed,Paypal,2534.49,844.83,3,2024-06-07,Overnight,Accessory,19.68



📋 COLUMN NAMES (16 total):
    1. Customer ID
    2. Age
    3. Gender
    4. Loyalty Member
    5. Product Type
    6. SKU
    7. Rating
    8. Order Status
    9. Payment Method
   10. Total Price
   11. Unit Price
   12. Quantity
   13. Purchase Date
   14. Shipping Type
   15. Add-ons Purchased
   16. Add-on Total

🔍 QUICK DATA QUALITY CHECK:
   • All columns present: 
   • No completely empty columns: 
   • Customer IDs look numeric: 


## 🔍 Step 5: Check Missing/Null Values
**Goal**: Detect incomplete records early

In [34]:
# Step 5: Check for missing/null values
print(" Step 5 COMPLETED: Missing Values Analysis")
print("=" * 60)

# Calculate missing values
missing_counts = df.isnull().sum()
missing_percentages = (df.isnull().sum() / len(df)) * 100

# Create missing values summary
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing_counts.values,
    'Missing_Percentage': missing_percentages.values
}).sort_values('Missing_Count', ascending=False)

print(" MISSING VALUES SUMMARY:")
print(missing_summary.to_string(index=False))

# Identify columns with missing data
columns_with_missing = missing_summary[missing_summary['Missing_Count'] > 0]
if not columns_with_missing.empty:
    print(f"\n⚠️  COLUMNS WITH MISSING DATA ({len(columns_with_missing)}):")
    for _, row in columns_with_missing.iterrows():
        print(f"   • {row['Column']}: {row['Missing_Count']} ({row['Missing_Percentage']:.1f}%)")
else:
    print("\n NO MISSING VALUES DETECTED - Dataset is complete!")

# Check for empty strings or whitespace-only values
print(f"\n🔍 CHECKING FOR EMPTY STRINGS...")
empty_strings_found = False
for col in df.select_dtypes(include=['object']).columns:
    empty_count = (df[col].str.strip() == '').sum()
    if empty_count > 0:
        print(f"   • {col}: {empty_count} empty strings")
        empty_strings_found = True

if not empty_strings_found:
    print("    No empty strings detected in text columns")

# Overall data completeness
total_cells = df.shape[0] * df.shape[1]
missing_cells = df.isnull().sum().sum()
completeness = ((total_cells - missing_cells) / total_cells) * 100

print(f"\n📈 OVERALL DATA COMPLETENESS: {completeness:.2f}%")
print(f"   • Total cells: {total_cells:,}")
print(f"   • Missing cells: {missing_cells:,}")
print(f"   • Complete cells: {total_cells - missing_cells:,}")

 Step 5 COMPLETED: Missing Values Analysis
 MISSING VALUES SUMMARY:
           Column  Missing_Count  Missing_Percentage
Add-ons Purchased           4868              24.340
           Gender              1               0.005
              Age              0               0.000
      Customer ID              0               0.000
     Product Type              0               0.000
              SKU              0               0.000
           Rating              0               0.000
   Loyalty Member              0               0.000
     Order Status              0               0.000
   Payment Method              0               0.000
       Unit Price              0               0.000
      Total Price              0               0.000
         Quantity              0               0.000
    Purchase Date              0               0.000
    Shipping Type              0               0.000
     Add-on Total              0               0.000

⚠️  COLUMNS WITH MISSING DATA 

## 🔄 Step 6: Check for Duplicates
**Goal**: Remove repeated rows if any

In [35]:
# Step 6: Check for duplicate records
print(" Step 6 COMPLETED: Duplicate Analysis")
print("=" * 60)

# Check for exact duplicates (all columns identical)
total_duplicates = df.duplicated().sum()
print(f"🔄 EXACT DUPLICATES: {total_duplicates}")

if total_duplicates > 0:
    print(f"   • Duplicate rows found: {total_duplicates}")
    print(f"   • Percentage of duplicates: {(total_duplicates/len(df))*100:.2f}%")
    
    # Show a sample of duplicated rows
    duplicate_rows = df[df.duplicated(keep=False)].sort_values(df.columns.tolist())
    print(f"\n📋 SAMPLE OF DUPLICATE ROWS:")
    display(duplicate_rows.head(6))
    
    # Option to remove duplicates
    print(f"\n⚠️  Consider removing duplicates in preprocessing step")
else:
    print("    No exact duplicate rows found!")

# Check for potential duplicates based on key identifiers
print(f"\n🔍 CHECKING KEY FIELD DUPLICATES:")

# Check Customer ID + Purchase Date combinations (potential same transaction)
if 'Customer ID' in df.columns and 'Purchase Date' in df.columns:
    customer_date_dups = df.duplicated(subset=['Customer ID', 'Purchase Date']).sum()
    print(f"   • Customer ID + Purchase Date duplicates: {customer_date_dups}")

# Check for multiple transactions per customer
customer_counts = df['Customer ID'].value_counts()
customers_with_multiple = (customer_counts > 1).sum()
max_transactions = customer_counts.max()

print(f"\n👥 CUSTOMER TRANSACTION PATTERNS:")
print(f"   • Unique customers: {df['Customer ID'].nunique():,}")
print(f"   • Customers with multiple transactions: {customers_with_multiple:,}")
print(f"   • Maximum transactions per customer: {max_transactions}")
print(f"   • Average transactions per customer: {customer_counts.mean():.2f}")

# Top customers by transaction count
if customers_with_multiple > 0:
    print(f"\n🏆 TOP 5 CUSTOMERS BY TRANSACTION COUNT:")
    top_customers = customer_counts.head()
    for customer_id, count in top_customers.items():
        print(f"   • Customer {customer_id}: {count} transactions")

 Step 6 COMPLETED: Duplicate Analysis
🔄 EXACT DUPLICATES: 0
    No exact duplicate rows found!

🔍 CHECKING KEY FIELD DUPLICATES:
   • Customer ID + Purchase Date duplicates: 33

👥 CUSTOMER TRANSACTION PATTERNS:
   • Unique customers: 12,136
   • Customers with multiple transactions: 5,499
   • Maximum transactions per customer: 8
   • Average transactions per customer: 1.65

🏆 TOP 5 CUSTOMERS BY TRANSACTION COUNT:
   • Customer 18304: 8 transactions
   • Customer 16357: 7 transactions
   • Customer 7070: 6 transactions
   • Customer 2238: 6 transactions
   • Customer 4224: 6 transactions


## 📅 Step 7: Validate Date Parsing
**Goal**: Ensure "Purchase Date" is recognized as datetime

In [36]:
# Step 7: Validate date parsing
print(" Step 7 COMPLETED: Date Parsing Validation")
print("=" * 60)

# Check current data type of Purchase Date
print(f"📅 CURRENT PURCHASE DATE INFO:")
print(f"   • Data type: {df['Purchase Date'].dtype}")
print(f"   • Sample values:")
for i, date_val in enumerate(df['Purchase Date'].head(3)):
    print(f"     {i+1}. {date_val}")

# Attempt to parse dates
try:
    # Convert to datetime
    df['Purchase Date'] = pd.to_datetime(df['Purchase Date'], format='%Y-%m-%d')
    
    print(f"\n DATE PARSING SUCCESSFUL!")
    print(f"   • New data type: {df['Purchase Date'].dtype}")
    
    # Extract date range
    min_date = df['Purchase Date'].min()
    max_date = df['Purchase Date'].max()
    date_range_days = (max_date - min_date).days
    
    print(f"\n DATE RANGE ANALYSIS:")
    print(f"   • Earliest date: {min_date.strftime('%Y-%m-%d (%A)')}")
    print(f"   • Latest date: {max_date.strftime('%Y-%m-%d (%A)')}")
    print(f"   • Date range: {date_range_days} days ({date_range_days/365:.1f} years)")
    
    # Check for any invalid dates or outliers
    current_date = pd.Timestamp.now()
    future_dates = df['Purchase Date'] > current_date
    
    if future_dates.any():
        print(f"⚠️  WARNING: {future_dates.sum()} future dates detected!")
    else:
        print(f" No future dates detected")
        
    # Monthly distribution
    df['Month'] = df['Purchase Date'].dt.month
    df['Year'] = df['Purchase Date'].dt.year
    monthly_counts = df['Purchase Date'].dt.to_period('M').value_counts().sort_index()
    
    print(f"\n📈 MONTHLY DISTRIBUTION:")
    print(f"   • Total months covered: {len(monthly_counts)}")
    print(f"   • Average transactions per month: {monthly_counts.mean():.0f}")
    print(f"   • Peak month: {monthly_counts.idxmax()} ({monthly_counts.max()} transactions)")
    print(f"   • Lowest month: {monthly_counts.idxmin()} ({monthly_counts.min()} transactions)")
    
except Exception as e:
    print(f"❌ DATE PARSING FAILED: {e}")
    print("   • Attempting alternative date formats...")
    
    # Try different date formats
    date_formats = ['%m/%d/%Y', '%d/%m/%Y', '%Y/%m/%d', '%m-%d-%Y', '%d-%m-%Y']
    parsed = False
    
    for fmt in date_formats:
        try:
            df['Purchase Date'] = pd.to_datetime(df['Purchase Date'], format=fmt, errors='coerce')
            if not df['Purchase Date'].isnull().all():
                print(f" Parsed with format: {fmt}")
                parsed = True
                break
        except:
            continue
    
    if not parsed:
        print("❌ Could not parse dates with common formats")
        print("   • Manual date format investigation required")

 Step 7 COMPLETED: Date Parsing Validation
📅 CURRENT PURCHASE DATE INFO:
   • Data type: object
   • Sample values:
     1. 2024-03-20
     2. 2024-04-20
     3. 2023-10-17

 DATE PARSING SUCCESSFUL!
   • New data type: datetime64[ns]

 DATE RANGE ANALYSIS:
   • Earliest date: 2023-09-24 (Sunday)
   • Latest date: 2024-09-23 (Monday)
   • Date range: 365 days (1.0 years)
 No future dates detected

📈 MONTHLY DISTRIBUTION:
   • Total months covered: 13
   • Average transactions per month: 1538
   • Peak month: 2024-01 (2049 transactions)
   • Lowest month: 2023-09 (190 transactions)


## 🎯 Phase 1 Summary: Data Overview Complete

**All 7 steps completed successfully!**

 **Task Checklist Status:**
-  **Step 1**: Data loaded successfully  
-  **Step 2**: Dataset shape analyzed  
-  **Step 3**: Column types identified  
-  **Step 4**: Data preview completed  
-  **Step 5**: Missing values checked  
-  **Step 6**: Duplicates analyzed  
-  **Step 7**: Date parsing validated  

**Ready for Phase 2: Exploratory Data Analysis (EDA)**

##  Figure Saving Examples

When you create visualizations, use the `save_figure()` function to automatically save them:

```python
# Example 1: Save a histogram
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df['Total Price'], bins=30, alpha=0.7)
ax.set_title('Distribution of Total Price')
save_figure(fig, 'price_distribution', category='distributions')

# Example 2: Save correlation heatmap  
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', ax=ax)
save_figure(fig, 'correlation_heatmap', category='correlations')

# Example 3: Save ROC curve
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.2f})')
save_figure(fig, 'roc_curve_model', category='model_performance')
```

**Figures will be saved as:**
-  `outputs/figures/distributions/price_distribution.png`
-  `outputs/figures/correlations/correlation_heatmap.png` 
-  `outputs/figures/model_performance/roc_curve_model.png`